# Week 13, Day 3: Constrained optimisation and Lagrange multipliers

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w013/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Projected gradient ascent is power iteration

Solve the constrained problem numerically: take a gradient step on u'Su, then project back onto the unit circle by normalising. Repeat 200 times from (0, 1) with step 0.1. Store the result in `u_star` and the multiplier u'Su in `lam_star`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
S = np.array([[3.0, 1.2], [1.2, 1.0]])
u = np.array([0.0, 1.0])
for _ in range(200):
    u = ...           # gradient step on u'Su (the gradient is 2Su)
    u = ...           # project back onto the unit circle
u_star = u
lam_star = ...
print("u* =", u_star.round(4), " lambda =", round(lam_star, 4), " eigenvalues:", np.linalg.eigvalsh(S).round(4))

In [ ]:
import numpy as np
def test_top_eigvec():
    "projected gradient ascent finds the top eigenvector (up to sign)"
    top = np.linalg.eigh(S)[1][:, -1]
    assert abs(abs(u_star @ top) - 1) < 1e-6

def test_multiplier():
    "the multiplier u'Su equals the largest eigenvalue"
    assert abs(lam_star - np.linalg.eigvalsh(S).max()) < 1e-6

test_top_eigvec()
test_multiplier()
print("All checks passed")

### Exercise: Correlated noise and the best readout

Forty orientation-tuned neurons have correlated noise that is strongest between neurons with similar preferences. Build the optimal readout `w_opt` from the formula and a naive readout `w_naive` proportional to f' (scaled to be unbiased). Store both variances in `var_opt` and `var_naive`.

In [ ]:
N = 40
prefs = np.linspace(0, np.pi, N, endpoint=False)
theta0 = np.pi / 3
def f(t): return 5 + 20 * np.exp(2 * (np.cos(2 * (t - prefs)) - 1))
fp = (f(theta0 + 1e-4) - f(theta0 - 1e-4)) / 2e-4            # tuning-curve slopes
d = np.abs(np.subtract.outer(prefs, prefs)); d = np.minimum(d, np.pi - d)
rate = f(theta0)
C = np.diag(0.5 * rate) + 0.5 * np.sqrt(np.outer(rate, rate)) * np.exp(-d / 1.0)

w_opt = ...
w_naive = ...
var_opt, var_naive = ..., ...
print(f"optimal variance {var_opt:.5f}, naive {var_naive:.5f}, ratio {var_naive / var_opt:.3f}")

In [ ]:
import numpy as np
def test_unbiased():
    "both readouts satisfy the constraint w'f' = 1"
    assert abs(w_opt @ fp - 1) < 1e-8 and abs(w_naive @ fp - 1) < 1e-8

def test_formula():
    "the optimal variance equals 1 / (f' C^-1 f')"
    assert abs(var_opt - 1 / (fp @ np.linalg.solve(C, fp))) < 1e-10

def test_better():
    "the optimal readout has lower variance than the readout that ignores correlations"
    assert var_opt < var_naive

test_unbiased()
test_formula()
test_better()
print("All checks passed")

### Exercise: Equalise the histogram

Draw 20,000 skewed contrasts. Compare a linear response (scaled so the 99.5th percentile maps to 1) with the empirical CDF as response. Quantise each into ten levels and compute the entropy in bits. Store `r_lin`, `r_cdf`, `H_lin` and `H_cdf`.

In [ ]:
rng = np.random.default_rng(1981)
x = rng.gamma(2.0, 1.0, 20000)             # many small contrasts, few large

def entropy10(r):
    p = np.bincount(np.minimum((r * 10).astype(int), 9), minlength=10) / len(r)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

r_lin = ...
r_cdf = ...                                # the rank of each x divided by the number of samples
H_lin, H_cdf = entropy10(r_lin), entropy10(r_cdf)
print(f"linear {H_lin:.3f} bits, CDF {H_cdf:.3f} bits, maximum {np.log2(10):.3f}")
plt.plot(np.sort(x), np.sort(r_cdf), label="CDF response"); plt.plot(np.sort(x), np.sort(r_lin), label="linear")
plt.hist(x, 80, density=True, alpha=0.3, label="input density"); plt.legend(); plt.xlabel("contrast"); plt.show()

In [ ]:
import numpy as np
def test_cdf_uniform():
    "the CDF response uses all ten levels almost equally"
    p = np.bincount(np.minimum((r_cdf * 10).astype(int), 9), minlength=10) / len(r_cdf)
    assert np.all(np.abs(p - 0.1) < 0.005)

def test_entropy_gain():
    "the CDF response has higher entropy than the linear one, and is close to the maximum log2(10)"
    assert H_cdf > H_lin + 0.3 and abs(H_cdf - np.log2(10)) < 0.01

test_cdf_uniform()
test_entropy_gain()
print("All checks passed")